### Question 1 — environment check

In [1]:
!uv --version

uv 0.13.5 (46776a95d 2026-10-21 x86_64-pc-windows-msvc)


### Question 2

```scikit-learn==1.7.2```

### Question 3

In [2]:
import pickle

lead = {
    "lead_source": "paid_ads",
    "industry": "technology",
    "employment_status": "employed",
    "location": "north_america",
    "number_of_courses_viewed": 2,
    "annual_income": 79276.0,
    "interaction_count": 4,
    "lead_score": 0.41,
}

with open("pipeline.bin", "rb") as f:
    model = pickle.load(f)

probabilities = model.predict_proba([lead])
print("Probabilities:", probabilities)
print("Conversion probability:", round(probabilities[0][1], 3))

Probabilities: [[0.46700558 0.53299442]]
Conversion probability: 0.533


## Question 4

In [3]:
import subprocess

project = "C:/Users/Admin/Downloads/Machine-Learning/HomeWork5"
uv = "C:/Users/Admin/.local/bin/uv.exe"

server = subprocess.Popen(
    [
        uv, "run", "uvicorn", "predict:app",
        "--host", "127.0.0.1",
        "--port", "9696"
    ],
    cwd=project,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

print("API process started:", server.pid)

API process started: 17720


In [7]:
import requests

response = requests.get("http://127.0.0.1:9696/health")
print(response.status_code)
print(response.json())

200
{'status': 'ok', 'model_sha256': '1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1', 'feature_defaults_sha256': '1694f842b908fac9f53c0cc19be6d91f934befc63af23cc32fefafde69acd0cd'}


In [5]:
lead = {
    "lead_source": "organic_search",
    "industry": "technology",
    "employment_status": "employed",
    "location": "europe",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0,
    "interaction_count": 7,
    "lead_score": 0.74,
}

response = requests.post(
    "http://127.0.0.1:9696/predict",
    json=lead
)

print(response.status_code)
result = response.json()
print(result)
print("Conversion probability:", round(result["conversion_probability"], 3))
print("Conversion:", result["conversion"])

200
{'conversion_probability': 0.769799, 'conversion': True}
Conversion probability: 0.77
Conversion: True


In [6]:
server.terminate()
server.wait()
print("API stopped")

API stopped


### Docker image

In [8]:
import subprocess

project = "C:/Users/Admin/Downloads/Machine-Learning/HomeWork5"

result = subprocess.run(
    ["docker", "build", "-t", "zoomcamp-model:2026-hw5", "."],
    cwd=project,
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("Docker build failed")


#0 building with "desktop-linux" instance using docker driver

#1 [internal] load build definition from Dockerfile
#1 transferring dockerfile: 637B 0.1s done
#1 DONE 0.1s

#2 [internal] load metadata for docker.io/library/python:3.11.15-slim-bookworm@sha256:d29f48a31a8b408ed19272ca1e7b10ebae13b240a27e862d3d4217c528e2e0c3
#2 ...

#3 [internal] load metadata for ghcr.io/astral-sh/uv:0.10.11@sha256:3472e43b4e738cf911c99d41bb34331280efad54c73b1def654a6227bb59b2b4
#3 DONE 1.7s

#2 [internal] load metadata for docker.io/library/python:3.11.15-slim-bookworm@sha256:d29f48a31a8b408ed19272ca1e7b10ebae13b240a27e862d3d4217c528e2e0c3
#2 DONE 1.9s

#4 [internal] load .dockerignore
#4 transferring context: 2B done
#4 DONE 0.0s

#5 [internal] load build context
#5 transferring context: 297B 0.0s done
#5 DONE 0.0s

#6 FROM ghcr.io/astral-sh/uv:0.10.11@sha256:3472e43b4e738cf911c99d41bb34331280efad54c73b1def654a6227bb59b2b4
#6 resolve ghcr.io/astral-sh/uv:0.10.11@sha256:3472e43b4e738cf911c99d41bb3433128

In [9]:
import subprocess
import time

container = subprocess.Popen(
    ["docker", "run", "--rm", "-p", "9696:9696",
     "zoomcamp-model:2026-hw5"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(3)
print("Container process running:", container.poll() is None)

Container process running: True


In [11]:
import requests

response = requests.get("http://localhost:9696/health")
print(response.status_code)
print(response.json())

200
{'status': 'ok', 'model_sha256': '1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1', 'feature_defaults_sha256': '1694f842b908fac9f53c0cc19be6d91f934befc63af23cc32fefafde69acd0cd'}


In [10]:
lead = {
    "lead_source": "organic_search",
    "industry": "technology",
    "employment_status": "employed",
    "location": "europe",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0,
    "interaction_count": 7,
    "lead_score": 0.74,
}

response = requests.post(
    "http://localhost:9696/predict",
    json=lead
)

print(response.status_code)

result = response.json()
print(result)
print("Conversion probability:", round(result["conversion_probability"], 3))
print("Conversion:", result["conversion"])

200
{'conversion_probability': 0.769799, 'conversion': True}
Conversion probability: 0.77
Conversion: True


In [12]:
container_id = subprocess.check_output(
    ["docker", "ps", "-q", "--filter", "ancestor=zoomcamp-model:2026-hw5"],
    text=True
).strip()

print(container_id)

a12ff5a4b064


In [13]:
subprocess.run(["docker", "stop", "a12ff5a4b064"], capture_output=True, text=True)

CompletedProcess(args=['docker', 'stop', 'a12ff5a4b064'], returncode=0, stdout='a12ff5a4b064\n', stderr='')

## Question 5

In [14]:
from pathlib import Path

project = Path("C:/Users/Admin/Downloads/Machine-Learning/HomeWork5")

In [15]:
dockerfile = project / "Dockerfile"

print(dockerfile.read_text())

FROM python:3.11.15-slim-bookworm@sha256:d29f48a31a8b408ed19272ca1e7b10ebae13b240a27e862d3d4217c528e2e0c3

COPY --from=ghcr.io/astral-sh/uv:0.10.11@sha256:3472e43b4e738cf911c99d41bb34331280efad54c73b1def654a6227bb59b2b4 /uv /uvx /bin/

ENV PATH="/code/.venv/bin:$PATH" \
    UV_COMPILE_BYTECODE=1 \
    UV_LINK_MODE=copy

WORKDIR /code
COPY pyproject.toml uv.lock .python-version ./
RUN uv sync --locked --no-dev

COPY model.py predict.py feature_defaults.json model_metadata.json pipeline.bin q6_test.py ./

EXPOSE 9696
ENTRYPOINT ["uvicorn", "predict:app", "--host", "0.0.0.0", "--port", "9696"]



```python:3.11.15-slim-bookworm```

## Question 6

In [16]:
import subprocess

run = subprocess.run(
    [
        "docker", "run", "-d", "--rm",
        "-p", "9696:9696",
        "zoomcamp-model:2026-hw5"
    ],
    capture_output=True,
    text=True
)

print("Container ID:", run.stdout.strip())
print("Error:", run.stderr)

container_id = run.stdout.strip()

Container ID: 85ab6179cbb7fda450ce5b3f5d752d0c3814b84deb906364d6049f3cd326bc68
Error: 


In [17]:
import requests
import time

for attempt in range(15):
    try:
        health = requests.get(
            "http://localhost:9696/health",
            timeout=2
        )
        if health.ok:
            break
    except requests.RequestException:
        time.sleep(1)

print(health.status_code)
print(health.json())

200
{'status': 'ok', 'model_sha256': '1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1', 'feature_defaults_sha256': '1694f842b908fac9f53c0cc19be6d91f934befc63af23cc32fefafde69acd0cd'}


In [18]:
project = "C:/Users/Admin/Downloads/Machine-Learning/HomeWork5"

result = subprocess.run(
    ["python", "q6_test.py"],
    cwd=project,
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

{'conversion_probability': 0.769799, 'conversion': True}




In [19]:
client = {
    "lead_source": "organic_search",
    "industry": "technology",
    "employment_status": "employed",
    "location": "europe",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0,
    "interaction_count": 7,
    "lead_score": 0.74,
}

response = requests.post(
    "http://localhost:9696/predict",
    json=client,
    timeout=10
)

response.raise_for_status()
result = response.json()

print(result)
print("Q6 answer:", round(result["conversion_probability"], 3))

{'conversion_probability': 0.769799, 'conversion': True}
Q6 answer: 0.77


In [20]:
subprocess.run(
    ["docker", "stop", "85ab6179cbb7"],
    capture_output=True,
    text=True
)

CompletedProcess(args=['docker', 'stop', '85ab6179cbb7'], returncode=0, stdout='85ab6179cbb7\n', stderr='')